<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/DevSimExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualizing DEVSIM results in a Colab Notebook
Source: https://github.com/devsim/devsim/blob/main/examples/plotting/visualization.ipynb

In [ ]:
!pip install -q devsim pyvista panel wurlitzer
!apt install libgl1-mesa-glx xvfb
!wget https://raw.githubusercontent.com/devsim/devsim/35ea1c121da5c8901bf7aef79e101183e1b89a9c/examples/diode/diode_common.py -O diode_common.py
!wget https://raw.githubusercontent.com/devsim/devsim/35ea1c121da5c8901bf7aef79e101183e1b89a9c/examples/diode/gmsh_diode2d.msh
!wget https://raw.githubusercontent.com/devsim/devsim/35ea1c121da5c8901bf7aef79e101183e1b89a9c/examples/diode/gmsh_diode3d.msh

In [ ]:
import devsim
try:
  devsim.delete_mesh(mesh="MyDevice")
except:
  pass
try:
  devsim.delete_mesh(mesh="dio")
except:
  pass
try:
  devsim.delete_device(device="MyDevice")
except:
  pass
try:
  devsim.delete_device(device="diode2d")
except:
  pass
try:
  devsim.delete_mesh(mesh="diode2d")
except:
  pass
try:
  devsim.delete_mesh(mesh="diode3d")
except:
  pass
try:
  devsim.delete_device(device="diode3d")
except:
  pass


# 1D: https://github.com/devsim/devsim/blob/35ea1c121da5c8901bf7aef79e101183e1b89a9c/examples/diode/diode_1d.py

In [ ]:
# Copyright 2013 DEVSIM LLC
#
# SPDX-License-Identifier: Apache-2.0

from devsim import *
import devsim.python_packages.simple_physics as simple_physics
import diode_common
#####
# dio1
#
# Make doping a step function
# print dat to text file for viewing in grace
# verify currents analytically
# in dio2 add recombination
#

device="MyDevice"
region="MyRegion"

diode_common.CreateMesh(device=device, region=region)

diode_common.SetParameters(device=device, region=region)
set_parameter(device=device, region=region, name="taun", value=1e-8)
set_parameter(device=device, region=region, name="taup", value=1e-8)

diode_common.SetNetDoping(device=device, region=region)

print_node_values(device=device, region=region, name="NetDoping")

diode_common.InitialSolution(device, region)

# Initial DC solution
solve(type="dc", absolute_error=1.0, relative_error=1e-10, maximum_iterations=30)

diode_common.DriftDiffusionInitialSolution(device, region)
###
### Drift diffusion simulation at equilibrium
###
solve(type="dc", absolute_error=1e10, relative_error=1e-10, maximum_iterations=30)


####
#### Ramp the bias to 0.5 Volts
####
v = 0.0
while v < 0.51:
    set_parameter(device=device, name=simple_physics.GetContactBiasName("top"), value=v)
    solve(type="dc", absolute_error=1e10, relative_error=1e-10, maximum_iterations=30)
    simple_physics.PrintCurrents(device, "top")
    simple_physics.PrintCurrents(device, "bot")
    v += 0.1

write_devices(file="diode_1d.dat", type="tecplot")

In [ ]:
import matplotlib
import matplotlib.pyplot
x=get_node_model_values(device=device, region=region, name="x")
ymax = 10
ymin = 10
fields = ("Electrons", "Holes", "Donors", "Acceptors")
for i in fields:
   y=get_node_model_values(device=device, region=region, name=i)
   if (max(y) > ymax):
     ymax = max(y)
   matplotlib.pyplot.semilogy(x, y)
matplotlib.pyplot.xlabel('x (cm)')
matplotlib.pyplot.ylabel('Density (#/cm^3)')
matplotlib.pyplot.legend(fields)
ymax *= 10
matplotlib.pyplot.axis([min(x), max(x), ymin, ymax])


In [ ]:
matplotlib.pyplot.clf()
edge_average_model(device=device, region=region, node_model="x", edge_model="xmid")
xmid=get_edge_model_values(device=device, region=region, name="xmid")
efields = ("ElectronCurrent", "HoleCurrent", )
y=get_edge_model_values(device=device, region=region, name="ElectronCurrent")
ymin=min(y)
ymax=max(y)
for i in efields:
 y=get_edge_model_values(device=device, region=region, name=i)
 if min(y) < ymin:
   ymin = min(y)
 elif max(y) > ymax:
   ymax = max(y)
 matplotlib.pyplot.plot(xmid, y)
matplotlib.pyplot.xlabel('x (cm)')
matplotlib.pyplot.ylabel('J (A/cm^2)')
matplotlib.pyplot.legend(efields)
matplotlib.pyplot.axis([min(x), max(x), 0.5*ymin, 2*ymax])

https://github.com/devsim/devsim/issues/88

# 2D

In [ ]:
# Copyright 2013 DEVSIM LLC
#
# SPDX-License-Identifier: Apache-2.0

from devsim import *
from devsim.python_packages.simple_physics import *
import diode_common

device="diode2d"
region="Bulk"

diode_common.Create2DGmshMesh(device, region)

# this is is the devsim format
write_devices    (file="gmsh_diode2d_out.msh")

diode_common.SetParameters(device=device, region=region)

####
#### NetDoping
####
node_model(device=device, region=region, name="Acceptors", equation="1.0e18*step(0.5e-5-y);")
node_model(device=device, region=region, name="Donors"   , equation="1.0e18*step(y-0.5e-5);")
node_model(device=device, region=region, name="NetDoping", equation="Donors-Acceptors;")

diode_common.InitialSolution(device, region)


####
#### Initial DC solution
####
solve(type="dc", absolute_error=1e100, relative_error=1e-12, maximum_iterations=30)

###
### Drift diffusion simulation at equilibrium
###
diode_common.DriftDiffusionInitialSolution(device, region)

solve(type="dc", absolute_error=1e10, relative_error=1e-10, maximum_iterations=50)

v = 0.0
while v < 0.51:
    set_parameter(device=device, name=GetContactBiasName("top"), value=v)
    solve(type="dc", absolute_error=1e10, relative_error=1e-10, maximum_iterations=30)
    PrintCurrents(device, "top")
    PrintCurrents(device, "bot")
    v += 0.1

write_devices(file="gmsh_diode2d.dat", type="tecplot")
write_devices(file="gmsh_diode2d_dd.msh", type="devsim")


In [ ]:
import pyvista as pv
from wurlitzer import pipes # To suppress some vtk output

In [ ]:
pv.start_xvfb()

In [ ]:
with pipes() as (out, err):
    reader = pv.get_reader('gmsh_diode2d.dat')
    mesh = reader.read()
    plotter = pv.Plotter(notebook=True)
    _ = plotter.add_mesh(mesh, scalars='Electrons', log_scale=True, cmap='RdBu')
    _ = plotter.show_grid()
    _ = plotter.camera_position = "xy"
    _ = plotter.show(jupyter_backend='panel')

#3D

In [ ]:
# Copyright 2013 DEVSIM LLC
#
# SPDX-License-Identifier: Apache-2.0

from devsim import *
from devsim.python_packages.simple_physics import *
import diode_common

device="diode3d"
region="Bulk"


diode_common.Create3DGmshMesh(device, region)

# this is is the devsim format
write_devices (file="gmsh_diode3d_out.msh")

diode_common.SetParameters(device=device, region=region)

####
#### NetDoping
####
node_model(device=device, region=region, name="Acceptors", equation="1.0e18*step(0.5e-5-z);")
node_model(device=device, region=region, name="Donors",    equation="1.0e18*step(z-0.5e-5);")
node_model(device=device, region=region, name="NetDoping", equation="Donors-Acceptors;")

diode_common.InitialSolution(device, region)


####
#### Initial DC solution
####
solve(type="dc", absolute_error=1e100, relative_error=1e-8, maximum_iterations=100)

###
### Drift diffusion simulation at equilibrium
###
diode_common.DriftDiffusionInitialSolution(device, region)

solve(type="dc", absolute_error=1e10, relative_error=1e-8, maximum_iterations=50)

v = 0.1
while v < 0.51:
    set_parameter(device=device, name=GetContactBiasName("top"), value=v)
    solve(type="dc", absolute_error=1e10, relative_error=1e-8, maximum_iterations=30)
    PrintCurrents(device, "top")
    PrintCurrents(device, "bot")
    v += 0.1

element_from_edge_model(edge_model="ElectricField",   device=device, region=region)
element_from_edge_model(edge_model="ElectronCurrent", device=device, region=region)
element_from_edge_model(edge_model="HoleCurrent",     device=device, region=region)

write_devices(file="gmsh_diode3d_dd.dat", type="devsim")
write_devices(file="gmsh_diode3d_dd.msh", type="devsim")

#element_from_node_model(node_model="node_index", device=device, region=region)
#en0 = map(lambda x : int(x), get_element_model_values(name="node_index@en0", device=device, region=region))
#en1 = map(lambda x : int(x), get_element_model_values(name="node_index@en1", device=device, region=region))
#en2 = map(lambda x : int(x), get_element_model_values(name="node_index@en2", device=device, region=region))
#en3 = map(lambda x : int(x), get_element_model_values(name="node_index@en3", device=device, region=region))
#for i in range(len(en0)):
#  print "%d %d %d %d" % (en0[i], en1[i], en2[i], en3[i])


In [ ]:
with pipes() as (out, err):
    reader = pv.get_reader('gmsh_diode3d_dd.dat')
    mesh = reader.read()
    plotter = pv.Plotter(notebook=True)
    _ = plotter.add_mesh(mesh, scalars='Electrons', log_scale=True, cmap='RdBu')
    _ = plotter.show_grid()
    _ = plotter.show(jupyter_backend='panel')